# Train YOLO11n — Fingers
Chạy lần lượt các ô. Bật GPU trong Runtime → Change runtime type. Kết quả được lưu vào Google Drive. Mỗi lần chạy ô train mới sẽ tạo một thư mục riêng.


In [ ]:
%pip install -q ultralytics roboflow


In [ ]:
from google.colab import drive
import torch
from pathlib import Path

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Hãy bật GPU trong Runtime → Change runtime type."
print(torch.cuda.get_device_name(0))


## Tải dataset
Nhập API key Roboflow khi được hỏi. Key được nhập ẩn. Workspace, project và version lấy từ notebook trước.


In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("hius-workspace-llhmr").project("fingers-ql10t")
dataset = project.version(1).download("yolov11")


In [ ]:
import yaml

root = Path(dataset.location).resolve()
config = yaml.safe_load((root / "data.yaml").read_text())
config["path"] = str(root)
for split, folder in [("train", "train"), ("val", "valid"), ("test", "test")]:
    images = root / folder / "images"
    if images.is_dir():
        config[split] = str(images)
    elif split == "test":
        config.pop("test", None)
    else:
        raise FileNotFoundError(images)
print("Classes:", config["names"])
for split in ("train", "val", "test"):
    if split in config:
        print(split, len(list(Path(config[split]).glob("*"))), "images")
data_yaml = root / "data_colab.yaml"
data_yaml.write_text(yaml.safe_dump(config, allow_unicode=True))


## Train mới
YOLO11n, 100 epochs, 640 px. Checkpoint lưu vào Drive sau mỗi epoch theo mặc định. Giữ runtime kết nối trong khi train.


In [ ]:
from ultralytics import YOLO
from datetime import datetime

run_name = "yolo11n_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
model = YOLO("yolo11n.pt")
model.train(
    data=str(data_yaml), epochs=100, imgsz=640, batch=16,
    device=0, workers=2, save=True,
    project="/content/drive/MyDrive/Fingers_YOLO/runs", name=run_name
)
print("Kết quả:", model.trainer.save_dir)


## Xem kết quả
Chạy sau khi train xong.


In [ ]:
from IPython.display import display, Image

run_dir = Path(model.trainer.save_dir)
for name in ["results.png", "confusion_matrix.png"]:
    path = run_dir / name
    if path.exists():
        display(Image(filename=str(path)))
print("Best:", run_dir / "weights" / "best.pt")
print("Last:", run_dir / "weights" / "last.pt")


## Chạy tiếp nếu bị ngắt
Chỉ chạy ô dưới khi lượt train đã dừng. Trong phiên mới, chạy lại các ô cài đặt, mount Drive và tải dataset trước. Dán đúng đường dẫn last.pt của lượt muốn tiếp tục. Không chạy lại ô train mới.


In [ ]:
# Bỏ dấu # ở các dòng dưới để resume lượt train bị ngắt.
# checkpoint = Path(input("Đường dẫn last.pt trên Drive: ").strip())
# assert checkpoint.is_file(), checkpoint
# model = YOLO(str(checkpoint))
# model.train(resume=True, data=str(data_yaml), device=0)
